In [1]:
import os
import subprocess

print("OpenAI API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Tavily API key loaded:", bool(os.getenv("TAVILY_API_KEY")))

node_version = subprocess.run(
    ["node", "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print("Node.js:", node_version.stdout.strip())

check_sdk = """
import * as pi from "@earendil-works/pi-coding-agent";
console.log("createAgentSession:", typeof pi.createAgentSession);
console.log("SessionManager:", typeof pi.SessionManager);
"""

sdk_check = subprocess.run(
    ["node", "--input-type=module", "-e", check_sdk],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(sdk_check.stdout)

if sdk_check.returncode != 0:
    print(sdk_check.stderr)
    raise RuntimeError("Pi SDK import failed")

OpenAI API key loaded: True
Tavily API key loaded: True
Node.js: v22.23.3
createAgentSession: function
SessionManager: function



In [2]:
import subprocess

js_smoke_test = r"""
import process from "node:process";
import {
  createAgentSession,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";

if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");

if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  thinkingLevel: "low",
});

try {
  await session.prompt("Reply with exactly: PI_AGENT_SDK_OK");
  console.log("Pi response:", session.getLastAssistantText());
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--input-type=module", "-e", js_smoke_test],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi smoke test failed with exit code {run.returncode}")

Pi response: PI_AGENT_SDK_OK



In [6]:
import subprocess

js_benchmark = r"""
import fs from "node:fs/promises";
import process from "node:process";
import { performance } from "node:perf_hooks";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";

const cases = JSON.parse(
  await fs.readFile("/workspace/shared/benchmark_cases.json", "utf8")
);
const benchmark = cases.long_prompt_math;

if (!benchmark) {
  throw new Error("Benchmark case 'long_prompt_math' was not found");
}
if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace",
  agentDir: getAgentDir(),
  systemPromptOverride: () =>
    "Follow every requirement in the user request. Do not use tools.",
  appendSystemPromptOverride: () => [],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  noTools: true,
  thinkingLevel: "medium",
});

try {
  const startedAt = performance.now();
  await session.prompt(benchmark.prompt);
  const latencyS = (performance.now() - startedAt) / 1000;

  const answer = session.getLastAssistantText() ?? "";
  const assistantMessages = session.messages.filter(
    (message) => message.role === "assistant"
  );
  const usages = assistantMessages
    .map((message) => message.usage)
    .filter(Boolean);

  const usage = usages.reduce(
    (total, item) => ({
      requests: total.requests + 1,
      input_tokens: total.input_tokens + item.input,
      output_tokens: total.output_tokens + item.output,
      reasoning_tokens: total.reasoning_tokens + (item.reasoning ?? 0),
      total_tokens: total.total_tokens + item.totalTokens,
    }),
    {
      requests: 0,
      input_tokens: 0,
      output_tokens: 0,
      reasoning_tokens: 0,
      total_tokens: 0,
    }
  );

  const runId =
    "pi-long-math-" +
    new Date().toISOString().replace(/[-:]/g, "").replace(/\.\d{3}Z$/, "Z");

  const record = {
    sdk: "pi-agent-sdk",
    case: "long_prompt_math",
    run_id: runId,
    model: `${model.provider}/${model.id}`,
    thinking_level: "medium",
    tools_enabled: false,
    latency_s: Number(latencyS.toFixed(2)),
    usage,
    requirements: benchmark.requirements,
    run_item_types: [...new Set(session.messages.map((message) => message.role))],
    answer,
  };

  const resultsDir = "/workspace/shared/results";
  await fs.mkdir(resultsDir, { recursive: true });

  const outputPath = `${resultsDir}/${runId}.json`;
  await fs.writeFile(outputPath, JSON.stringify(record, null, 2), "utf8");

  console.log("Answer:\\n" + answer);
  console.log("\\nRun ID:", runId);
  console.log("Latency (seconds):", record.latency_s);
  console.log("Usage:", JSON.stringify(usage));
  console.log("Saved:", outputPath);
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--input-type=module", "-e", js_benchmark],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi benchmark failed with exit code {run.returncode}")

Answer:\n## Executive summary

Using only the **supplied test data, not externally verified facts**, revenue grew **20.0%** from FY2024 to FY2025, and net income grew **27.5%**. Net margin increased from **8.0%** to **8.5%**. FY2024 net debt was **THB 200 million**. FY2025 net debt is **unavailable** because cash was not reported.

## Financial table

All amounts and figures below are based on **supplied test data, not externally verified facts**. Amounts are in **million THB**, except percentages.

| Metric | FY2024 | FY2025 |
|---|---:|---:|
| Revenue | 1,000 | 1,200 |
| Revenue growth | — | 20.0% |
| Net income | 80 | 102 |
| Net income growth | — | 27.5% |
| Net margin | 8.0% | 8.5% |
| Total debt | 300 | 280 |
| Cash | 100 | Not reported |
| Net debt | 200 | Unavailable |

## Calculation notes

All inputs and results are **supplied test data or calculations based on supplied test data, not externally verified facts**.

- **Revenue growth:** \((\text{FY2025 revenue} - \text{FY2024 

In [7]:
import json
import statistics
from pathlib import Path

results_dir = Path("/workspace/shared/results")
exploratory_run_id = "pi-long-math-20260929T094439Z"

records = []

for path in sorted(results_dir.glob("pi-long-math-*.json")):
    if path.stem == exploratory_run_id:
        continue

    record = json.loads(path.read_text(encoding="utf-8"))
    if record.get("case") == "long_prompt_math":
        records.append(record)

if len(records) < 3:
    raise ValueError(f"Expected 3 standard runs; found {len(records)}")

records = records[-3:]
latencies = [record["latency_s"] for record in records]
tokens = [record["usage"]["total_tokens"] for record in records]

for record in records:
    print(
        f'{record["run_id"]} | '
        f'latency={record["latency_s"]:.2f}s | '
        f'tokens={record["usage"]["total_tokens"]}'
    )

print("Median latency:", round(statistics.median(latencies), 2), "seconds")
print("Latency range:", round(min(latencies), 2), "to", round(max(latencies), 2), "seconds")
print("Median total tokens:", statistics.median(tokens))

pi-long-math-20260929T094836Z | latency=6.16s | tokens=1023
pi-long-math-20260929T095035Z | latency=6.37s | tokens=1048
pi-long-math-20260929T095045Z | latency=6.81s | tokens=1059
Median latency: 6.37 seconds
Latency range: 6.16 to 6.81 seconds
Median total tokens: 1048


In [10]:
import subprocess

js_mcp_check = r"""
import fs from "node:fs/promises";
import process from "node:process";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";

if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}
if (!process.env.TAVILY_API_KEY) {
  throw new Error("TAVILY_API_KEY is missing");
}

// เขียน config ใน Pi agent directory ภายใน container
const agentDir = getAgentDir();
await fs.mkdir(agentDir, { recursive: true });

const mcpConfig = {
  mcpServers: {
    tavily: {
      url: "https://mcp.tavily.com/mcp/",
      headers: {
        Authorization: "Bearer ${TAVILY_API_KEY}",
      },
      exposure: "direct",
      timeout: 60,
    },
  },
};

await fs.writeFile(
  `${agentDir}/mcp.json`,
  JSON.stringify(mcpConfig, null, 2),
  "utf8"
);

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace/pi_agent",
  agentDir,
  additionalExtensionPaths: ["builtin:mcp"],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  thinkingLevel: "off",
});

try {
  await session.bindExtensions({});

  console.log("Active tools:", session.getActiveToolNames());
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--input-type=module", "-e", js_mcp_check],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi MCP check failed with exit code {run.returncode}")

Active tools: [ 'read', 'bash', 'edit', 'write' ]



In [11]:
import subprocess

check = subprocess.run(
    ["/workspace/node_modules/.bin/pi", "mcp", "list"],
    cwd="/workspace/pi_agent",
    capture_output=True,
    text=True,
)

print("STDOUT:")
print(check.stdout)

print("STDERR:")
print(check.stderr)

print("Exit code:", check.returncode)

STDOUT:
No MCP-related files/configs found under `/workspace/pi_agent` (searched up to 3 levels).

I also don’t have any MCP tool namespace exposed in this session, so there are no connected MCP servers/tools to list from here.

STDERR:

Exit code: 0


In [12]:
import json
import subprocess
from pathlib import Path

config_path = Path("/workspace/pi_agent/.pi/mcp.json")
config_path.parent.mkdir(parents=True, exist_ok=True)

config = {
    "mcpServers": {
        "tavily": {
            "url": "https://mcp.tavily.com/mcp/",
            "headers": {
                "Authorization": "Bearer ${TAVILY_API_KEY}"
            },
            "exposure": "direct",
            "timeout": 60
        }
    }
}

config_path.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print("Created:", config_path)

check = subprocess.run(
    ["/workspace/node_modules/.bin/pi", "mcp", "list"],
    cwd="/workspace/pi_agent",
    capture_output=True,
    text=True,
)

print("\nSTDOUT:")
print(check.stdout)
print("STDERR:")
print(check.stderr)
print("Exit code:", check.returncode)

Created: /workspace/pi_agent/.pi/mcp.json

STDOUT:
Configured MCP servers in `.pi/mcp.json`:

- `tavily`
  - URL: `https://mcp.tavily.com/mcp/`
  - Auth: `Bearer ${TAVILY_API_KEY}`
  - Exposure: `direct`
  - Timeout: `60s`

STDERR:

Exit code: 0


In [13]:
import subprocess

js_mcp_check = r"""
import process from "node:process";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";

if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}
if (!process.env.TAVILY_API_KEY) {
  throw new Error("TAVILY_API_KEY is missing");
}

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace/pi_agent",
  agentDir: getAgentDir(),
  additionalExtensionPaths: ["builtin:mcp"],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  thinkingLevel: "off",
});

try {
  await session.bindExtensions({});

  // รอให้การเชื่อมต่อ MCP ซึ่งเริ่มทำงานเบื้องหลังมีเวลาโหลด tools
  const deadline = Date.now() + 20000;
  let mcpTools = [];

  while (Date.now() < deadline) {
    mcpTools = session
      .getActiveToolNames()
      .filter((name) => name.startsWith("mcp__tavily__"));

    if (mcpTools.length > 0) break;

    await new Promise((resolve) => setTimeout(resolve, 500));
  }

  console.log("Active tools:", session.getActiveToolNames());
  console.log("Tavily tools:", mcpTools);

  if (mcpTools.length === 0) {
    throw new Error("No Tavily tools became active within 20 seconds");
  }
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--input-type=module", "-e", js_mcp_check],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi MCP check failed with exit code {run.returncode}")

Active tools: [ 'read', 'bash', 'edit', 'write' ]
Tavily tools: []

file:///workspace/[eval1]:63
    throw new Error("No Tavily tools became active within 20 seconds");
          ^

Error: No Tavily tools became active within 20 seconds
    at file:///workspace/[eval1]:63:11

Node.js v22.23.3



RuntimeError: Pi MCP check failed with exit code 1

In [14]:
print("Exit code:", run.returncode)
print("STDOUT:")
print(run.stdout or "(empty)")
print("STDERR:")
print(run.stderr or "(empty)")

Exit code: 1
STDOUT:
Active tools: [ 'read', 'bash', 'edit', 'write' ]
Tavily tools: []

STDERR:
file:///workspace/[eval1]:63
    throw new Error("No Tavily tools became active within 20 seconds");
          ^

Error: No Tavily tools became active within 20 seconds
    at file:///workspace/[eval1]:63:11

Node.js v22.23.3



In [15]:
import subprocess

js_check_exports = r"""
import * as pi from "@earendil-works/pi-coding-agent";

console.log("MCP exports:", Object.keys(pi).filter((name) => /mcp/i.test(name)));
console.log(
  "Extension exports:",
  Object.keys(pi).filter((name) => /extension/i.test(name))
);
console.log(
  "Resource loader exports:",
  Object.keys(pi).filter((name) => /ResourceLoader|AgentDir/i.test(name))
);
"""

check = subprocess.run(
    ["node", "--input-type=module", "-e", js_check_exports],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print("STDOUT:")
print(check.stdout or "(empty)")
print("STDERR:")
print(check.stderr or "(empty)")
print("Exit code:", check.returncode)

STDOUT:
MCP exports: []
Extension exports: [
  'ExtensionEditorComponent',
  'ExtensionInputComponent',
  'ExtensionRunner',
  'ExtensionSelectorComponent',
  'createExtensionRuntime',
  'discoverAndLoadExtensions'
]
Resource loader exports: [ 'DefaultResourceLoader', 'getAgentDir' ]

STDERR:
(empty)
Exit code: 0


In [1]:
import subprocess

js_mcp_check = r"""
import process from "node:process";
import {
  createAgentSession,
  DefaultResourceLoader,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";
import { createMcpAdapter } from "pi-mcp-adapter";

if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}
if (!process.env.TAVILY_API_KEY) {
  throw new Error("TAVILY_API_KEY is missing");
}

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const mcpExtension = createMcpAdapter({
  config: {
    mcpServers: {
      tavily: {
        url: "https://mcp.tavily.com/mcp/",
        headers: {
          Authorization: `Bearer ${process.env.TAVILY_API_KEY}`,
        },
        lifecycle: "eager",
        directTools: true,
        requestTimeoutMs: 60000,
      },
    },
  },
});

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace/pi_agent",
  extensionFactories: [mcpExtension],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  thinkingLevel: "off",
});

try {
  await session.bindExtensions({});

  const deadline = Date.now() + 30000;
  let activeTools = [];

  while (Date.now() < deadline) {
    activeTools = session.getActiveToolNames();

    if (activeTools.some((name) => name.toLowerCase().includes("tavily"))) {
      break;
    }

    await new Promise((resolve) => setTimeout(resolve, 500));
  }

  console.log("Active tools:", activeTools);
  console.log(
    "Tavily tools:",
    activeTools.filter((name) => name.toLowerCase().includes("tavily"))
  );
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--import", "tsx", "--input-type=module", "-e", js_mcp_check],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print("STDOUT:")
print(run.stdout or "(empty)")
print("STDERR:")
print(run.stderr or "(empty)")
print("Exit code:", run.returncode)

STDOUT:
(empty)
STDERR:
node:internal/modules/package_json_reader:314
  throw new ERR_MODULE_NOT_FOUND(packageName, fileURLToPath(base), null);
        ^

Error [ERR_MODULE_NOT_FOUND]: Cannot find package 'typebox' imported from /workspace/node_modules/pi-mcp-adapter/index.ts
    at Object.getPackageJSONURL (node:internal/modules/package_json_reader:314:9)
    at packageResolve (node:internal/modules/esm/resolve:772:81)
    at moduleResolve (node:internal/modules/esm/resolve:859:18)
    at defaultResolve (node:internal/modules/esm/resolve:989:11)
    at #cachedDefaultResolve (node:internal/modules/esm/loader:747:20)
    at #resolveAndMaybeBlockOnLoaderThread (node:internal/modules/esm/loader:783:38)
    at nextStep (node:internal/modules/customization_hooks:189:26)
    at resolveBaseSync (file:///workspace/node_modules/tsx/dist/register-nyXW-TH3.mjs:2:11092)
    at resolveDirectorySync (file:///workspace/node_modules/tsx/dist/register-nyXW-TH3.mjs:2:12398)
    at resolveTsPathsSync (fi